This notebook contains descriptive data analysis and pre-processing and application of ML techniques to predict hospital infection by fungus or bacteria among COVID-19 positeve patients.


In [ ]:
# Última modificação: 10/02/2025

In [3]:
!pip install tensorflow
!pip install shap
!pip install seaborn
!pip install imblearn
!pip install 'openpyxl>=3.0.0'

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [3]:
# Libraries
import numpy as np
import pandas as pd
import random as rd
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime as dt
import os
import time
import math

from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression, SelectKBest, SelectPercentile
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, make_scorer, roc_auc_score, recall_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.under_sampling import RandomUnderSampler
from sklearn.impute import KNNImputer, SimpleImputer



# 1.0 Read original dataset

In [44]:
ls

COVID-PROGNOSTIC.csv*  hosp1_sen.csv  hosp2_sen.csv  ML_notebook.ipynb
HCAI-INFECTION.xlsx*   hosp1_spe.csv  hosp2_spe.csv  README.md
hosp1_auc.csv          hosp2_auc.csv  LICENSE


In [45]:
original_dataset = pd.read_excel("HCAI-INFECTION.xlsx")

original_dataset = original_dataset.drop(columns='Unnamed: 0')

original_dataset

,ID_PACIENTE,Idade,ALT (TGP),Basófilos,Bilirrubina Direta,Bilirrubina Indireta,CHCM,CK,Calcio Ionizavel,Creatinina,...,RDW,Sódio,TP_INR,TTPA - Paciente_Normal,Uréia,VCM,Volume plaquetário médio,SEXO,INFEC,Infecção Hospitalar
0,004688799FD293C3ABE0A07209FD8B75,69,32.0,10.0,0.21,0.15,32.4,194.0,1.17,2.29,...,13.8,138.0,0.99,0.94,91.0,96.0,10.6,1,0,negativo
1,009F0D6B3BA6C0E2D406585697D679EB,57,25.0,10.0,0.19,0.15,34.8,153.0,NaN,1.22,...,13.1,137.0,0.97,1.11,38.0,89.5,10.3,1,0,negativo
2,0183BA4D9368936BAD131398B55CDDC3,69,142.0,10.0,0.19,0.14,32.8,106.0,1.24,1.20,...,12.7,138.0,1.00,0.95,50.0,84.4,10.8,1,0,negativo
3,01A30B6624DDB49F16CA4311CC37D65F,38,25.0,10.0,0.10,0.09,33.8,60.0,NaN,0.98,...,13.0,142.0,NaN,NaN,25.0,83.0,9.4,1,0,negativo
4,02367C393B8487123744A46CFBB91A28,74,13.0,10.0,0.18,0.12,33.1,72.0,1.14,0.80,...,13.1,135.0,1.10,1.20,39.0,94.5,9.7,1,0,negativo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,FE69345DBB6AE27A1E81775EE898A25B,65,23.0,10.0,0.25,0.27,34.1,225.0,1.26,0.77,...,13.0,138.0,1.30,1.03,26.0,89.7,11.1,1,0,negativo
495,FECC5CE1CFE3BCE881F29C2333527135,42,10.0,40.0,NaN,NaN,33.1,NaN,NaN,0.86,...,12.9,139.0,NaN,NaN,29.0,92.5,10.4,0,0,negativo
496,FF19A1D8C1EB3A7A73541F3443B4FA00,79,20.0,40.0,0.21,0.22,32.5,170.0,1.22,0.70,...,15.2,142.0,1.20,1.05,27.0,99.2,11.4,0,0,negativo
497,FF4B2EED093AE641B9328FDB293C4116,53,40.0,0.0,0.16,0.25,33.3,NaN,1.20,1.05,...,12.7,140.0,0.97,1.06,25.0,84.1,11.0,1,0,negativo


In [46]:
original_dataset.columns

Index(['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'Basófilos', 'Bilirrubina Direta',
       'Bilirrubina Indireta', 'CHCM', 'CK', 'Calcio Ionizavel', 'Creatinina',
       'DHL', 'Dimeros D, quant', 'Eosinófilos', 'Eritrócitos, urina',
       'Fibrinogenio', 'Fosfatase Alcalina', 'Gama-GT', 'Glicose', 'HCM',
       'HCO3 venoso', 'Hemoglobina', 'Leucócitos', 'Leucócitos, urina',
       'Linfócitos', 'Magnésio', 'Monócitos', 'Neutrófilos', 'Plaquetas',
       'Potássio', 'Proteína C-Reativa', 'RDW', 'Sódio', 'TP_INR',
       'TTPA - Paciente_Normal', 'Uréia', 'VCM', 'Volume plaquetário médio',
       'SEXO', 'INFEC', 'Infecção Hospitalar'],
      dtype='object')

In [47]:
original_dataset.columns = ['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'Basófilos', 'Bilirrubina Direta',
       'Bilirrubina Indireta', 'CHCM', 'CK', 'Calcio Ionizavel', 'Creatinina',
       'DHL', 'Dimeros D, quant', 'Eosinófilos', 'Eritrócitos, urina',
       'Fibrinogenio', 'Fosfatase Alcalina', 'Gama-GT', 'Glicose', 'HCM',
       'HCO3 venoso', 'Hemoglobina', 'Leucócitos', 'Leucócitos, urina',
       'Linfócitos', 'Magnésio', 'Monócitos', 'Neutrófilos', 'Plaquetas',
       'Potássio', 'Proteína C-Reativa', 'RDW', 'Sódio', 'TP_INR',
       'TTPA - Paciente_Normal', 'Uréia', 'VCM', 'Volume plaquetário médio',
       'SEXO', 'TARGET', 'Infecção Hospitalar']

original_dataset

,ID_PACIENTE,Idade,ALT (TGP),Basófilos,Bilirrubina Direta,Bilirrubina Indireta,CHCM,CK,Calcio Ionizavel,Creatinina,...,RDW,Sódio,TP_INR,TTPA - Paciente_Normal,Uréia,VCM,Volume plaquetário médio,SEXO,TARGET,Infecção Hospitalar
0,004688799FD293C3ABE0A07209FD8B75,69,32.0,10.0,0.21,0.15,32.4,194.0,1.17,2.29,...,13.8,138.0,0.99,0.94,91.0,96.0,10.6,1,0,negativo
1,009F0D6B3BA6C0E2D406585697D679EB,57,25.0,10.0,0.19,0.15,34.8,153.0,NaN,1.22,...,13.1,137.0,0.97,1.11,38.0,89.5,10.3,1,0,negativo
2,0183BA4D9368936BAD131398B55CDDC3,69,142.0,10.0,0.19,0.14,32.8,106.0,1.24,1.20,...,12.7,138.0,1.00,0.95,50.0,84.4,10.8,1,0,negativo
3,01A30B6624DDB49F16CA4311CC37D65F,38,25.0,10.0,0.10,0.09,33.8,60.0,NaN,0.98,...,13.0,142.0,NaN,NaN,25.0,83.0,9.4,1,0,negativo
4,02367C393B8487123744A46CFBB91A28,74,13.0,10.0,0.18,0.12,33.1,72.0,1.14,0.80,...,13.1,135.0,1.10,1.20,39.0,94.5,9.7,1,0,negativo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494,FE69345DBB6AE27A1E81775EE898A25B,65,23.0,10.0,0.25,0.27,34.1,225.0,1.26,0.77,...,13.0,138.0,1.30,1.03,26.0,89.7,11.1,1,0,negativo
495,FECC5CE1CFE3BCE881F29C2333527135,42,10.0,40.0,NaN,NaN,33.1,NaN,NaN,0.86,...,12.9,139.0,NaN,NaN,29.0,92.5,10.4,0,0,negativo
496,FF19A1D8C1EB3A7A73541F3443B4FA00,79,20.0,40.0,0.21,0.22,32.5,170.0,1.22,0.70,...,15.2,142.0,1.20,1.05,27.0,99.2,11.4,0,0,negativo
497,FF4B2EED093AE641B9328FDB293C4116,53,40.0,0.0,0.16,0.25,33.3,NaN,1.20,1.05,...,12.7,140.0,0.97,1.06,25.0,84.1,11.0,1,0,negativo


In [48]:
dataset1 = original_dataset.copy(deep=True)

print(dataset1.shape)

dataset1.head(3)

(499, 40)


,ID_PACIENTE,Idade,ALT (TGP),Basófilos,Bilirrubina Direta,Bilirrubina Indireta,CHCM,CK,Calcio Ionizavel,Creatinina,...,RDW,Sódio,TP_INR,TTPA - Paciente_Normal,Uréia,VCM,Volume plaquetário médio,SEXO,TARGET,Infecção Hospitalar
0,004688799FD293C3ABE0A07209FD8B75,69,32.0,10.0,0.21,0.15,32.4,194.0,1.17,2.29,...,13.8,138.0,0.99,0.94,91.0,96.0,10.6,1,0,negativo
1,009F0D6B3BA6C0E2D406585697D679EB,57,25.0,10.0,0.19,0.15,34.8,153.0,NaN,1.22,...,13.1,137.0,0.97,1.11,38.0,89.5,10.3,1,0,negativo
2,0183BA4D9368936BAD131398B55CDDC3,69,142.0,10.0,0.19,0.14,32.8,106.0,1.24,1.20,...,12.7,138.0,1.00,0.95,50.0,84.4,10.8,1,0,negativo


In [49]:
dataset2 = pd.read_csv("COVID-PROGNOSTIC.csv")

dataset2 = dataset2.drop(columns="Unnamed: 0")

dataset2.columns

Index(['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'AST (TGO)', 'Basófilos',
       'Basófilos (%)', 'CHCM', 'Creatinina', 'Eosinófilos', 'Eosinófilos (%)',
       'Eritrócitos', 'HCM', 'Hematócrito', 'Hemoglobina', 'Leucócitos',
       'Linfócitos', 'Linfócitos (%)', 'Monócitos', 'Monócitos (%)',
       'Neutrófilos', 'Neutrófilos (%)', 'Plaquetas', 'Potássio',
       'Proteína C-Reativa', 'RDW', 'Sódio', 'Uréia', 'VCM',
       'Volume plaquetário médio', 'SEXO', 'GRAVIDADE'],
      dtype='object')

In [50]:
dataset2.columns = ['ID_PACIENTE', 'Idade', 'ALT (TGP)', 'AST (TGO)', 'Basófilos',
       'Basófilos (%)', 'CHCM', 'Creatinina', 'Eosinófilos', 'Eosinófilos (%)',
       'Eritrócitos', 'HCM', 'Hematócrito', 'Hemoglobina', 'Leucócitos',
       'Linfócitos', 'Linfócitos (%)', 'Monócitos', 'Monócitos (%)',
       'Neutrófilos', 'Neutrófilos (%)', 'Plaquetas', 'Potássio',
       'Proteína C-Reativa', 'RDW', 'Sódio', 'Uréia', 'VCM',
       'Volume plaquetário médio', 'SEXO', 'TARGET']

dataset2

,ID_PACIENTE,Idade,ALT (TGP),AST (TGO),Basófilos,Basófilos (%),CHCM,Creatinina,Eosinófilos,Eosinófilos (%),...,Plaquetas,Potássio,Proteína C-Reativa,RDW,Sódio,Uréia,VCM,Volume plaquetário médio,SEXO,TARGET
0,00017961865C4F766FDBB3CD8FE0BFB0,54,26.0,24.0,40.0,0.6,34.5,1.02,60.0,0.9,...,176000.0,4.0,0.11,13.1,138.0,35.0,86.0,9.8,1,0
1,000F0BC139D2846DB86AA32B8F05B215,41,NaN,NaN,20.0,0.4,33.2,1.04,160.0,3.4,...,279000.0,4.3,NaN,14.0,142.0,33.0,83.2,10.0,1,0
2,0028785949D91BD93442838FC898E229,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,002B919CC409B11DE52FB212379BE2CB,41,26.0,26.0,40.0,0.6,32.9,0.73,240.0,3.4,...,275000.0,NaN,0.12,13.3,NaN,30.0,88.3,11.2,0,0
4,003051C9B19101D1C10C5DC654384017,36,27.0,18.0,10.0,0.3,33.4,0.97,30.0,0.9,...,231000.0,4.2,0.08,14.4,139.0,16.0,83.7,10.4,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4315,FFABB233208E1E66DDC025BC5CC2E4D2,68,18.0,19.0,70.0,1.1,34.7,1.19,710.0,11.5,...,260000.0,NaN,0.13,12.1,NaN,53.0,85.3,9.9,1,0
4316,FFB440876DC059A0359FEFBA1D6FB28C,20,22.0,30.0,30.0,0.5,31.4,0.95,130.0,2.2,...,286000.0,3.6,0.03,12.7,138.0,28.0,85.3,11.3,1,0
4317,FFEC3898BAA04751EB00C108270B8F7E,39,35.0,23.0,40.0,0.4,34.5,0.91,190.0,2.1,...,147000.0,4.6,0.63,12.4,139.0,32.0,85.3,9.9,1,0
4318,FFF5753408C98D5E0218931420B6AF85,17,52.0,47.0,20.0,0.2,33.8,0.78,200.0,2.1,...,247000.0,4.0,0.07,13.2,139.0,22.0,87.3,11.2,0,0


In [51]:
dataset2["TARGET"].unique()

array([0, 1])

# 2.0 **MACHINE LEARNING**

In [ ]:
#dataset
hosp1 = dataset1
hosp2 = dataset2
data = {
    "hosp1": ( (hosp1.drop(columns=["Infecção Hospitalar", "TARGET", "ID_PACIENTE"])), hosp1.TARGET),
    "hosp2": ( (hosp2.drop(columns=["TARGET", "ID_PACIENTE"])), hosp2.TARGET)
}

#algorithms
algorithms = {
    "SVM": (SVC(probability= True), {"C": [1, 10], "kernel": ("linear", "rbf"), "gamma": ('scale', 'auto')}),
    "RF" : (RandomForestClassifier(random_state=0), {"n_estimators": [100,200,500], "max_depth": [4,6,10],"max_features":("auto", "sqrt")}),
    "GB" : (GradientBoostingClassifier(random_state=0), {"n_estimators": [100,200,500],  'learning_rate': [0.05, 0.1], "max_depth": [4,6,10]})
}


ini = time.time()

#3 folds to choose the best hyperparameters
gskf = StratifiedKFold(n_splits=3, shuffle=True, random_state=20) 

#choose of the best hyperparameters through balanced accuracy
perf = balanced_accuracy_score

#5-fold cross validation 
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=20) 

#define Standard Scaler to standardize the features
prep = StandardScaler()

#undersampling the majority class when classes are umbalanced
under = RandomUnderSampler(sampling_strategy='majority', random_state = 0)

#impute missing values with the mean of the 3 nearest neighbours
imputer = KNNImputer(n_neighbors=3, weights = 'distance')

#for each dataset
for name, (X,y) in data.items(): 

    #store the recall of each algorithm 
    score = {}
    for algorithm in algorithms.keys():
        score[algorithm] = []

    #store the auc for each algorithm
    auc_score = {}
    for algorithm in algorithms.keys():
        auc_score[algorithm] = []
    
    #for each algorithm and its respective search space
    for algorithm, (clf, parameters) in algorithms.items():
        
        #define a grid search for the best hyperparameters
        best = GridSearchCV(clf, parameters, cv=gskf, scoring=(make_scorer(perf)))

        for train, test in kf.split(X, y):
            
            #split train and test 
            X_train, X_test = X.iloc[train], X.iloc[test]
            y_train, y_test = y.iloc[train], y.iloc[test]
            
            #check if the majoritary class is 1.5 times larger than the other
            y_ = pd.DataFrame.from_dict(y)
            if (((y_[y_.TARGET == 1].shape[0])*1.5) < (y_[y_.TARGET == 0].shape[0])):
                
                #vectors to store y_pred e y_true
                y_pred = [] 
                y_true = [] 

                #undersampling of the majority class
                X_train, y_train = under.fit_resample(X_train, y_train)

                #impute missing values with the mean of the 3 nearest neighbors
                imputer.fit(X_train)
                X_train = imputer.transform(X_train)
                X_test = imputer.transform(X_test)        

                #standardize the features                
                prep.fit(X_train)

                #search for the best hyperparameters
                best.fit(prep.transform(X_train), y_train)

                #store the results
                y_pred = [*y_pred, *(best.predict(prep.transform(X_test)))] 
                y_true =  [*y_true, *y_test] 

                #calculate the recall
                score[algorithm].append(recall_score(y_true, y_pred, labels = [0,1], average = None))

                #calculate the area under roc curve
                aucscore = roc_auc_score(y_test, (best.predict_proba(prep.transform(X_test)))[:, 1])
                auc_score[algorithm].append(aucscore)

           
            #if classes are not umbalanced
            else:
                #vectors to store y_pred e y_true
                y_pred = [] 
                y_true = [] 

                #impute missing values with the mean of the 3 nearest neighbors
                imputer.fit(X_train)
                X_train = imputer.transform(X_train)
                X_test = imputer.transform(X_test)        

                #standardize the features                
                prep.fit(X_train)

                #search for the best hyperparameters
                best.fit(prep.transform(X_train), y_train)

                #store the results
                y_pred = [*y_pred, *(best.predict(prep.transform(X_test)))] 
                y_true =  [*y_true, *y_test] 

                #calculate the recall
                score[algorithm].append(recall_score(y_true, y_pred, labels = [0,1], average = None))

                #calculate the area under roc curve
                aucscore = roc_auc_score(y_test, (best.predict_proba(prep.transform(X_test)))[:, 1])
                auc_score[algorithm].append(aucscore)
    
    #write a csv with auc values
    auc_score = pd.DataFrame.from_dict(auc_score)  
    auc_score.to_csv(name + '_auc.csv')
    
    #write a csv with the recall of class '0' - specificity 
    #and another csv with the recall of class '1' - sensitivity
    recall_svm = pd.DataFrame(np.vstack(score['SVM']))
    recall_gb = pd.DataFrame(np.vstack(score['GB']))
    recall_rf = pd.DataFrame(np.vstack(score['RF']))

    esp = pd.concat([recall_svm[[0]], recall_rf[[0]], recall_gb[[0]]], axis=1)
    sen = pd.concat([recall_svm[[1]], recall_rf[[1]], recall_gb[[1]]], axis=1)

    esp.columns = ['SVM', 'RF', 'GB']
    sen.columns = ['SVM', 'RF', 'GB']

    esp.to_csv(name + '_spe.csv')
    sen.to_csv(name + '_sen.csv')  

fim = time.time()
print('\n')
print('Tempo de execução', round((fim - ini)/60, 4), 'minutos')

/home/filipe/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
27 fits failed out of a total of 54.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "/home/filipe/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/filipe/.local/lib/python3.10/site-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/home/filipe/.local/lib/python3.10/site-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/home/filipe/.local



Tempo de execução 24.7523 minutos


In [52]:
#dataset
hosp1 = dataset1
hosp2 = dataset2
data = {
    "hosp1": ( (hosp1.drop(columns=["Infecção Hospitalar", "TARGET", "ID_PACIENTE"])), hosp1.TARGET),
    "hosp2": ( (hosp2.drop(columns=["TARGET", "ID_PACIENTE"])), hosp2.TARGET)
}

#algorithms
algorithms = {
    "SVM": (SVC(probability= True), {"C": [1, 10], "kernel": ("linear", "rbf"), "gamma": ('scale', 'auto')}),
    "RF" : (RandomForestClassifier(random_state=0), {"n_estimators": [100,200,500], "max_depth": [4,6,10],"max_features":[10,20,30]}),
    "GB" : (GradientBoostingClassifier(random_state=0), {"n_estimators": [100,200,500],  'learning_rate': [0.05, 0.1], "max_depth": [4,6,10]})
}


# Definição de imputers para testar diferentes abordagens
imputers = {
    "KNN": KNNImputer(n_neighbors=3, weights='distance'),
    "Mean": SimpleImputer(strategy="mean"),
    "Median": SimpleImputer(strategy="median"),
    "MostFreq": SimpleImputer(strategy="most_frequent")
}

#3 folds to choose the best hyperparameters
gskf = StratifiedKFold(n_splits=3, shuffle=True, random_state=20) 

#choose of the best hyperparameters through balanced accuracy
perf = balanced_accuracy_score

#5-fold cross validation 
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=20) 

#define Standard Scaler to standardize the features
prep = StandardScaler()

#undersampling the majority class when classes are umbalanced
under = RandomUnderSampler(sampling_strategy='majority', random_state = 0)

ini = time.time()


# Loop sobre cada dataset
for name, (X, y) in data.items():  
    for imputer_name, imputer in imputers.items():  # Loop pelos imputers
        
        # Dicionários para armazenar os resultados por algoritmo
        score = {alg: [] for alg in algorithms.keys()}
        auc_score = {alg: [] for alg in algorithms.keys()}
    
        # Loop pelos algoritmos
        for algorithm, (clf, parameters) in algorithms.items():
            
            best = GridSearchCV(clf, parameters, cv=gskf, scoring=(make_scorer(perf)))

            for train, test in kf.split(X, y):
                
                X_train, X_test = X.iloc[train], X.iloc[test]
                y_train, y_test = y.iloc[train], y.iloc[test]
                
                y_ = pd.DataFrame.from_dict(y)
                if (((y_[y_.TARGET == 1].shape[0]) * 1.5) < (y_[y_.TARGET == 0].shape[0])):
                    
                    y_pred, y_true = [], []
                    
                    X_train, y_train = under.fit_resample(X_train, y_train)
                    
                    # Aplicação do imputer atual
                    imputer.fit(X_train)
                    X_train = imputer.transform(X_train)
                    X_test = imputer.transform(X_test)

                    prep.fit(X_train)
                    
                    best.fit(prep.transform(X_train), y_train)
                    
                    y_pred.extend(best.predict(prep.transform(X_test)))
                    y_true.extend(y_test) 

                    score[algorithm].append(recall_score(y_true, y_pred, labels=[0,1], average=None))
                    aucscore = roc_auc_score(y_test, (best.predict_proba(prep.transform(X_test)))[:, 1])
                    auc_score[algorithm].append(aucscore)

                else:
                    y_pred, y_true = [], []
                    
                    imputer.fit(X_train)
                    X_train = imputer.transform(X_train)
                    X_test = imputer.transform(X_test)

                    prep.fit(X_train)
                    
                    best.fit(prep.transform(X_train), y_train)
                    
                    y_pred.extend(best.predict(prep.transform(X_test)))
                    y_true.extend(y_test) 

                    score[algorithm].append(recall_score(y_true, y_pred, labels=[0,1], average=None))
                    aucscore = roc_auc_score(y_test, (best.predict_proba(prep.transform(X_test)))[:, 1])
                    auc_score[algorithm].append(aucscore)

        # Salvar os resultados para cada imputer
        result_prefix = f"{name}_{imputer_name}"  

        auc_df = pd.DataFrame.from_dict(auc_score)
        auc_df.to_csv(result_prefix + '_auc.csv')

        recall_svm = pd.DataFrame(np.vstack(score['SVM']))
        recall_gb = pd.DataFrame(np.vstack(score['GB']))
        recall_rf = pd.DataFrame(np.vstack(score['RF']))

        esp = pd.concat([recall_svm[[0]], recall_rf[[0]], recall_gb[[0]]], axis=1)
        sen = pd.concat([recall_svm[[1]], recall_rf[[1]], recall_gb[[1]]], axis=1)

        esp.columns = ['SVM', 'RF', 'GB']
        sen.columns = ['SVM', 'RF', 'GB']

        esp.to_csv(result_prefix + '_spe.csv')
        sen.to_csv(result_prefix + '_sen.csv')  

fim = time.time()
print('\n')
print('Tempo de execução', round((fim - ini)/60, 4), 'minutos')



/home/filipe/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
27 fits failed out of a total of 54.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
27 fits failed with the following error:
Traceback (most recent call last):
  File "/home/filipe/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/filipe/.local/lib/python3.10/site-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
  File "/home/filipe/.local/lib/python3.10/site-packages/sklearn/base.py", line 436, in _validate_params
    validate_parameter_constraints(
  File "/home/filipe/.local



Tempo de execução 98.0699 minutos


In [4]:
ls

COVID-PROGNOSTIC.csv*  LICENSE            README.md
HCAI-INFECTION.xlsx*   ML_notebook.ipynb  results/


In [24]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Pasta contendo os arquivos
results_folder = 'results/'
output_folder = 'output_excel/'
os.makedirs(output_folder, exist_ok=True)

# Lista de imputadores e algoritmos extraídos dos arquivos
imputadores = set()
algoritmos = ["SVM", "GB", "RF"]
hospitais = set()

for file in os.listdir(results_folder):
    if file.startswith("hosp") and "_auc.csv" in file:
        parts = file.split("_")
        hospitais.add(parts[0])  # Identifica os diferentes hospitais
        imputadores.add(parts[1])  # Considera apenas o tipo de imputador

# Dicionário para armazenar os resultados por hospital
df_hospitais = {}

# Criar um arquivo Excel para cada hospital
for hospital in sorted(hospitais):
    output_file = f'{output_folder}{hospital}_resultados.xlsx'
    with pd.ExcelWriter(output_file) as writer:
        df_hospitais[hospital] = {}
        
        for algoritmo in algoritmos:
            df_results = pd.DataFrame()
            
            for imputador in sorted(imputadores):
                hosp_prefix = f"{hospital}_{imputador}"  
                
                # Carregar os arquivos específicos para o imputador
                auc = pd.read_csv(f'{results_folder}{hosp_prefix}_auc.csv')
                sen = pd.read_csv(f'{results_folder}{hosp_prefix}_sen.csv')
                spe = pd.read_csv(f'{results_folder}{hosp_prefix}_spe.csv')
                
                # Criar DataFrame com média e desvio padrão lado a lado
                df_results[f'{imputador}_mean'] = [
                    auc[algoritmo].mean(),
                    sen[algoritmo].mean(),
                    spe[algoritmo].mean()
                ]
                df_results[f'{imputador}_std'] = [
                    auc[algoritmo].std(),
                    sen[algoritmo].std(),
                    spe[algoritmo].std()
                ]
            
            df_results.index = ["auc", "positive", "negative"]
            df_results.to_excel(writer, sheet_name=algoritmo)
            df_hospitais[hospital][algoritmo] = df_results

# Criar arquivo consolidado geral
output_file_geral = f'{output_folder}geral_resultados.xlsx'
with pd.ExcelWriter(output_file_geral) as writer:
    for algoritmo in algoritmos:
        df_geral_mean = pd.DataFrame()
        df_geral_std = pd.DataFrame()
        
        for hospital in sorted(hospitais):
            df_hospital = df_hospitais[hospital][algoritmo]
            
            df_geral_mean[f'{hospital}_mean'] = df_hospital.filter(like='_mean').mean(axis=1)
            df_geral_std[f'{hospital}_std'] = df_hospital.filter(like='_std').mean(axis=1)
        
        df_geral = pd.concat([df_geral_mean, df_geral_std], axis=1)
        df_geral.to_excel(writer, sheet_name=algoritmo)

In [25]:
# Gerar boxplots para cada hospital e para os resultados consolidados

boxplot_folder = "boxplot_folder/"

for hospital in sorted(hospitais) + ["geral"]:
    input_file = f'{output_folder}{hospital}_resultados.xlsx'
    for algoritmo in algoritmos:
        df_results = pd.read_excel(input_file, sheet_name=algoritmo, index_col=0)
        
        df_melted = df_results.T.reset_index()
        df_melted[['Imputador', 'Tipo']] = df_melted['index'].astype(str).str.rsplit('_', n=1, expand=True)
        df_melted = df_melted.melt(id_vars=['Imputador', 'Tipo'], var_name='Métrica', value_name='Valor')
        df_melted['Valor'] = pd.to_numeric(df_melted['Valor'], errors='coerce')
        
        plt.figure(figsize=(10, 6))
        sns.boxplot(x='Métrica', y='Valor', hue='Imputador', data=df_melted)
        plt.title(f'Boxplot para {algoritmo} - {hospital}')
        plt.xlabel('Métrica')
        plt.ylabel('Valor')
        plt.legend(title='Imputador')
        plt.xticks(rotation=45)
        plt.grid(True, linestyle='--', alpha=0.7)
        
        plt.savefig(f'{boxplot_folder}{hospital}_{algoritmo}_boxplot.png')
        plt.close()


# The end